# Code Interpreter Agent — Write, Execute, Self-Correct
## Write Code, Run It, Fix It — The Self-Correcting Code Loop
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/20-code-interpreter/code_interpreter_workbook.ipynb)

Builds a LangGraph agent that writes Python code with an LLM, runs it in a subprocess sandbox with a hard timeout, and self-corrects on error — looping up to MAX_ITERATIONS=3 times.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Code interpreter vs tool-calling agents; why subprocess sandboxing matters |
| 2 | **execute_code()** — `subprocess.run()` with TIMEOUT=10; capturing stdout/stderr/returncode |
| 3 | **Write Code Node** — LLM writes raw Python (no markdown); retry prompt includes previous code + error |
| 4 | **Conditional Exit** — `route()` → END if no error or MAX_ITERATIONS hit, else back to write_code |
| 5 | **Full Pipeline** — START → write_code → run_code → [write_code loop | END] |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
import subprocess, sys
from typing import Literal, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# MAX_ITERATIONS=3 and TIMEOUT=10 are the only levers: raise MAX_ITERATIONS for harder tasks, lower TIMEOUT for safety
MAX_ITERATIONS = 3
TIMEOUT = 10

SAMPLE_TASKS = [
    "Write a function that returns the first N Fibonacci numbers and print the first 10.",
    "Sort [(3,1),(1,4),(2,2),(4,3)] by second element descending and print.",
    "Check if 'racecar' and 'hello' are palindromes and print each result.",
]

# subprocess isolates execution: a crash or infinite loop in the generated code won't kill this process
# TIMEOUT=10 is the hard wall; subprocess.TimeoutExpired is caught and returned as an error string
def execute_code(code: str) -> dict:
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=TIMEOUT)
        return {"stdout": r.stdout, "stderr": r.stderr, "returncode": r.returncode}
    except subprocess.TimeoutExpired:
        return {"stdout": "", "stderr": f"Timed out after {TIMEOUT}s", "returncode": -1}

# TypedDict lets LangGraph access state keys with type hints; no runtime cost vs a dataclass
class CodeInterpreterState(TypedDict):
    task: str; code: str; output: str; error: str; iterations: int

llm = ChatOpenAI(model="gpt-4o-mini")

# Retry prompt feeds back the previous code + error; without context, the LLM regenerates the same bug
def write_code_node(state):
    if state["error"]:
        prompt = f"Task: {state['task']}\n\nYour code:\n{state['code']}\n\nError:\n{state['error']}\n\nFix it. Output ONLY raw Python — no markdown, no explanation."
    else:
        prompt = f"Task: {state['task']}\n\nWrite a complete, runnable Python script. Output ONLY raw Python — no markdown, no explanation."
    code = llm.invoke([SystemMessage(content="You are an expert Python programmer."), HumanMessage(content=prompt)]).content.strip()
    code = code.replace("```python", "").replace("```", "").strip()
    return {"code": code, "iterations": state["iterations"] + 1}

def run_code_node(state):
    r = execute_code(state["code"])
    if r["returncode"] == 0:
        return {"output": r["stdout"], "error": ""}
    return {"output": "", "error": r["stderr"] or r["stdout"]}

# MAX_ITERATIONS guard prevents infinite retry if the LLM repeatedly generates broken code
def route(state) -> Literal["write_code", "__end__"]:
    if not state["error"] or state["iterations"] >= MAX_ITERATIONS: return END
    return "write_code"

g = StateGraph(CodeInterpreterState)
g.add_node("write_code", write_code_node)
g.add_node("run_code", run_code_node)
g.add_edge(START, "write_code")
g.add_edge("write_code", "run_code")
# add_conditional_edges: route()'s return value ('write_code' or END) is the next node name
g.add_conditional_edges("run_code", route)
# compile() locks the graph topology — after this, no nodes or edges can be added
app = g.compile()

for task in SAMPLE_TASKS:
    init = {"task": task, "code": "", "output": "", "error": "", "iterations": 0}
    r = app.invoke(init)
    print(f"Task: {task}")
    print(f"Iterations: {r['iterations']}  |  Success: {not r['error']}")
    print(f"Output: {r['output'].strip()[:200]}")
    print()

print("=== Exercises ===")
print("Ex1: Add TIMEOUT=2 and test with an infinite loop task — does it time out correctly?")
print("Ex2: Add a 4th task that deliberately fails on first attempt (wrong syntax) to see self-correction.")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*